# ACCESSAI: First Proto

In [ ]:
from ultralytics import YOLO
import torch
import os
import yaml
import requests
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
print(f"¿GPU disponible? {torch.cuda.is_available()}")
print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
print(f"Versión HIP: {torch.version.hip}")

In [ ]:
DATASET_PATH = "dataset"  # Ajusta si es necesario

# 1. Verificar la estructura de carpetas
print("Estructura del dataset:")
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(DATASET_PATH, split)
    if os.path.exists(split_path):
        images_path = os.path.join(split_path, 'images')
        labels_path = os.path.join(split_path, 'labels')
        n_images = len(os.listdir(images_path)) if os.path.exists(images_path) else 0
        n_labels = len(os.listdir(labels_path)) if os.path.exists(labels_path) else 0
        print(f"  {split}: {n_images} imágenes, {n_labels} labels")
    else:
        print(f"  {split}: ❌ No existe")

# 2. Leer el data.yaml
yaml_path = os.path.join(DATASET_PATH, "data.yaml")
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("\nContenido del data.yaml:")
print(f"  nc: {data_config.get('nc')}")
print(f"  names: {data_config.get('names')}")
print(f"  train: {data_config.get('train')}")
print(f"  val: {data_config.get('val')}")
print(f"  test: {data_config.get('test')}")

In [ ]:
with open(os.path.join(DATASET_PATH, "data.yaml"), 'r') as f:
    data_config = yaml.safe_load(f)
    
class_names = data_config['names']

def contar_clases(labels_path):
    contador = Counter()
    for label_file in os.listdir(labels_path):
        if label_file.endswith('.txt'):
            with open(os.path.join(labels_path, label_file), 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:  # Formato YOLO Detección: clase x y w h
                        class_id = int(parts[0])
                        contador[class_id] += 1
                        
    return contador


conteo_train = contar_clases(os.path.join(DATASET_PATH, "train", "labels"))
conteo_val = contar_clases(os.path.join(DATASET_PATH, "valid", "labels"))
conteo_test = contar_clases(os.path.join(DATASET_PATH, "test", "labels"))


df_clases = pd.DataFrame({
    'Clase': class_names,
    'Train': [conteo_train.get(i, 0) for i in range(len(class_names))],
    'Valid': [conteo_val.get(i, 0) for i in range(len(class_names))],
    'Test': [conteo_test.get(i, 0) for i in range(len(class_names))]
})


df_clases['Total'] = df_clases['Train'] + df_clases['Valid'] + df_clases['Test']   # Total
df_clases['% del total'] = (df_clases['Total'] / df_clases['Total'].sum() * 100).round(2)   # Porcentaje


df_clases = df_clases.sort_values('Total', ascending=False).reset_index(drop=True)   # Sorting

print("DISTRIBUCIÓN DE CLASES EN EL DATASET")
print("=" * 70)
print(df_clases.to_string(index=False))
print("=" * 70)
print(f"Total de instancias: {df_clases['Total'].sum()}") # Total de instancias/bounding boxes/objetos detectados

#### IMPORTANTE: Habrá que reducir el número de clases y seleccionar aquellas que consideremos más importantes para la detección de barreras de accesibilidad urbana. 

In [ ]:
# Descarga del modelo preentrenado y validación inicial con el dataset
!yolo val model=yolo26n.pt data=dataset/data.yaml name=val_inicial

#### mAP: mean Average Precision, métrica estándar para evaluar modelos de detección de objetos, que mide cuán bueno es el modelo detectando objetos y localizándolos correctamente. maP50 a 0.0333, MUY BAJO, el modelo intenta detectar las clases de nuestro dataset pero solo conoce las clases mediante las que ha sido entrenado

In [ ]:
model = YOLO("yolo26n.pt")

In [ ]:
# Entenamiento base del modelo, punto de partida:

results = model.train(
    data="dataset/data.yaml",
    epochs=40,          
    imgsz=640,          
    batch=16,
    device=0    # Activa la GPU
)

### Métricas del entrenamiento

Durante el entrenamiento, Ultralytics muestra una serie de métricas que reflejan el rendimiento del modelo. Estas son las más importantes:

#### Pérdidas (Losses)

| Métrica | ¿Qué mide? | ¿Qué esperar? |
| :--- | :--- | :--- |
| **box_loss** | Error en la localización de las bounding boxes (coordenadas). | Debe bajar con las épocas. |
| **cls_loss** | Error en la clasificación de cada objeto (¿es un coche? ¿una persona?). | Debe bajar con las épocas. |
| **l1_loss** | Pérdida auxiliar de regresión que ayuda a estabilizar el entrenamiento. | Debe bajar con las épocas. |

#### Métricas de evaluación (Box)

| Métrica | ¿Qué mide? | ¿Qué esperar? |
| :--- | :--- | :--- |
| **Precision (P)** | De todos los objetos que el modelo detecta, ¿cuántos son correctos? | Cuanto más alto, mejor. Cercano a 1.0 es ideal. |
| **Recall (R)** | De todos los objetos reales, ¿cuántos detecta el modelo? | Cuanto más alto, mejor. Cercano a 1.0 es ideal. |
| **mAP50** | Precisión media con un umbral de solapamiento (IoU) del 50%. | Cuanto más alto, mejor. >0.90 es excelente. |
| **mAP50-95** | Precisión media promediada sobre umbrales de IoU de 50% a 95%. | Cuanto más alto, mejor. >0.70 es muy bueno. |

#### Otras métricas

| Métrica | ¿Qué mide? |
| :--- | :--- |
| **Instances** | Número de objetos anotados en el batch actual. |
| **GPU_mem** | Memoria de la GPU utilizada durante el entrenamiento. |
| **Size** | Tamaño de las imágenes de entrada (ej. 640x640). |

#### ¿Qué es el IoU?

El **IoU** (Intersection over Union) mide el solapamiento entre la bounding box predicha y la real. Va de 0 (sin solapamiento) a 1 (solapamiento perfecto). Un IoU de 0.5 significa que la predicción cubre al menos el 50% del objeto real.

#### ¿Qué es el mAP?

El **mAP** (mean Average Precision) es la métrica estándar para evaluar modelos de detección de objetos. Combina Precision y Recall en un solo número. Un mAP alto significa que el modelo detecta los objetos correctamente y con buena localización.

Si el mAP es bueno (>0.90) → Documentar y pasar a la demo.

Si el mAP es bajo (<0.80) → Analizar qué clases fallan y considerar:

    Más épocas.

    Data augmentation específico.

    Ajuste de hiperparámetros con model.tune().

Si la demo falla mucho → Documentar la limitación de dominio y proponer fine-tuning con imágenes locales como trabajo futuro.

In [ ]:
model = YOLO("runs/detect/train-2/weights/best.pt")  # Cargamos el mejor modelo

In [ ]:
metrics = model.val(data="dataset/data.yaml", split="test")  # Evaluamos

In [ ]:
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("runs/detect/train-2/confusion_matrix_normalized.png")  # Matriz de confusión normalizada
plt.figure(figsize=(14, 12))
plt.imshow(img)
plt.axis('off')
plt.show()

### Evaluación con imágenes propias

In [ ]:
from pathlib import Path
import cv2

model = YOLO("runs/detect/train-2/weights/best.pt")
print(model.names)
print(f"Total clases: {len(model.names)}")

imgs_dir = Path("tests/imgs/")
output_dir = Path("tests/results_imgs/")
output_dir.mkdir(exist_ok=True)

EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

for img_path in sorted(imgs_dir.iterdir()):
    if not img_path.is_file() or img_path.suffix.lower() not in EXTENSIONES:
        continue

    results = model(img_path, conf=0.2, device="cpu")  # conf = umbral de confianza
    r = results[0]

    print(f"\n--- {img_path.name} ---")
    if len(r.boxes) == 0:
        print("  (sin detecciones)") 
    else:
        for box in r.boxes:
            clase = model.names[int(box.cls)]
            conf = float(box.conf)
            xyxy = box.xyxy[0].tolist()
            print(f"  {clase:20s} conf={conf:.2f}  bbox={[round(v) for v in xyxy]}")

    annotated = r.plot()
    cv2.imwrite(str(output_dir / img_path.name), annotated)